# 2 · Discretize a well-defined Sobolev SOLA problem

Keep the same 600 rays, three physical targets, noise draw and response constraint.
Define the continuum space first, then approximate its objective with conforming
finite trial spaces. This is still a **fully finite quadratic problem** at each level.

We use $H=H^2_\ell(\Omega)$, $\ell=0.1$, with the restriction/Bessel-potential norm
implemented by `pg.Sobolev`. This is not the intrinsic derivative norm with
arbitrarily chosen boundary conditions. The original smooth truth belongs to H2.
Ray integrals and disk means are bounded on this space; no prior radius or
probability law is asserted.

In [ ]:
%matplotlib inline
from pathlib import Path
import hashlib, json, os, socket, subprocess, sys, time
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
import study as F
S=F.S
ex=F.Experiment(length_scale=.1)  # shared by both H2 routes; edit consistently in all notebooks
OUT=Path('output');OUT.mkdir(exist_ok=True)
COLORS=('#2563eb','#dc2626','#7c3aed')
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':.2,'font.size':11})
print('host:',socket.gethostname(),'Python:',sys.executable)
print('source commit:',subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True).stdout.strip())
print('original problem SHA256:',hashlib.sha256(Path(S.P.__file__).read_bytes()).hexdigest())
print('600 original rays; one fixed noise draw; sigma =',S.P.SIGMA)
for label,c,t in zip(F.LABELS,ex.centres,ex.truth):print(label,c,'radius',ex.radius,'true disk mean',f'{t:.9f}')
print('Fixed normalized trade-off lambda =',ex.lam,'; H2 length scale =',ex.length_scale)
import platform, scipy, pygeoinf0 as pg
from threadpoolctl import threadpool_info
runtime=dict(host=socket.gethostname(),python=sys.version,numpy=np.__version__,scipy=scipy.__version__,
             platform=platform.platform(),cpu_count=os.cpu_count(),package_source=pg.__file__,
             threadpools=threadpool_info(),targets=ex.centres,radius=ex.radius,lambda_value=ex.lam,length_scale=ex.length_scale,
             study_sha256=hashlib.sha256(Path(F.__file__).read_bytes()).hexdigest(),
             helper_sha256=hashlib.sha256(Path(S.__file__).read_bytes()).hexdigest(),
             problem_sha256=hashlib.sha256(Path(S.P.__file__).read_bytes()).hexdigest())
F.save(OUT/'runtime.json',runtime)
print('BLAS thread counts:',[v['num_threads'] for v in runtime['threadpools']])


## A conforming finite model realization

Use nested dyadic point grids $Z_h$ and trial functions $\phi_j=k_{z_j}$, the
H2 Riesz representers of point evaluation. These are trial centres, not extra data.
Their union is dense: a continuous H2 function orthogonal to every $k_z$ vanishes
on a dense set and therefore everywhere.

The finite matrices are
$$M_{jk}=\langle\phi_j,\phi_k\rangle_H,\quad
E_{ij}=G_i\phi_j,\quad U_{aj}=T_a\phi_j.$$
Thus
$$\boxed{K_h=E M^{-1}E^T,\qquad B_h=U M^{-1}E^T.}$$
The mass matrix is now a **full Sobolev Gram matrix**. Ordinary discontinuous
pixels would not be conforming H2 functions. The finite builder evaluates these
matrices without assembling the continuum ray/ray Gram.

The code below makes one small realization explicitly; the refinement loop uses
exactly this algebra.

In [ ]:
import pygeoinf0 as pg
H,F_ray,F_target=F.sobolev_objects(ex,32)
z=np.linspace(0,1,5);xx,yy=np.meshgrid(z,z)
Phi=H.riesz(np.column_stack([xx.ravel(),yy.ravel()]))
with S.quadrature_nodes(24):
    M=np.asarray(H.inner(Phi,Phi))
    E=np.asarray(H.inner(F_ray,Phi))
    U=np.asarray(H.inner(F_target,Phi))
factor=la.cho_factor(M,lower=True)
Z=la.cho_solve(factor,E.T)
K_h=E@Z;B_h=U@Z
print('M:',M.shape,'E:',E.shape,'U:',U.shape,'K_h:',K_h.shape)
print('No continuum G G* is required by this finite assembly.')


## Keep physical normalization and penalty fixed

The constraint is the exact physical response $v_i=G_i1=L_i$, known analytically.
It is an explicit constraint on the **data weights**, even though the trial space
need not contain the constant function exactly. Do not replace it silently with
$G_iP_h1$: that would change the constrained problem at each level.

The continuum normalized penalty $\tau=\lambda\|t_H\|_H^2$ is also kept fixed;
using $\|P_ht_H\|$ to rescale it on every mesh would change the objective.
A continuum-reference calculation is made **only to assess accuracy** and fix
that target norm, not to supply $K_h$ or $B_h$ to the finite builder.

In [ ]:
ref,Xref,tau,reference=F.native_reference(ex,24,32)
print('Fixed tau:',tau,'reference estimates:',reference['estimate'])
print('Reference cost is excluded from each finite method timing.')
X=F.constrained_sola(K_h,B_h,tau,ex.v)
print('Exact physical constant response:',X@ex.v)
projected_v=E@la.cho_solve(factor,np.ones(len(M)))
print('Projected trial-space response:',X@projected_v)
assert np.allclose(X@ex.v,1.,rtol=0,atol=2e-12)


## Refine nested trial spaces

$P_h$ is the H-orthogonal projector onto the trial space. Boundedness gives
$$K_h=GP_hG^*,\quad B_h=TP_hG^*.$$
Approximation of the data representers implies $K_h\to GG^*$, $B_h\to TG^*$.
The constrained minimizer converges as well: the positive fixed penalty gives
uniform coercivity, and $v$ is the same nonzero vector on every level.

We report both weight error and an actual functional error in the **common H2
geometry**, using $(X_h-X)K(X_h-X)^T$. This does not confuse Riesz functions with
ordinary area-density kernels.

In [ ]:
rows=[];weights=[]
max_n=int(os.environ.get('SOLA_PART1_MAX_TRIAL','65'))
for n in F.TRIALS:
    if n>max_n:continue
    system=F.galerkin_system(ex,n,24,32)
    X,row=F.solve_system(system,ex,tau)
    row.update(n=n,h=1/(n-1),dimension=n*n,**F.compare_to_reference(X,ref,Xref))
    row['gram_error']=float(la.norm(system['K']-ref['K'])/la.norm(ref['K']))
    row['projected_response_error']=float(np.max(np.abs(X@system['projected_v']-1)))
    rows.append(row);weights.append(X)
    print(f"{n:2d} x {n:2d}: relative X error={row['weight_error']:.3e}, "
          f"functional error={row['delivered_functional_error']:.3e}, estimates={np.round(row['estimate'],6)}")
F.save(OUT/'galerkin.json',dict(rows=rows,reference=reference,tau=tau))
np.savez_compressed(OUT/'galerkin_weights.npz',weights=weights,reference=Xref,grids=[r['n'] for r in rows])
assert rows[-1]['weight_error']<rows[0]['weight_error']
assert max(np.max(np.abs(np.array(r['constant_response'])-1)) for r in rows)<2e-10


In [ ]:
h=np.array([r['h'] for r in rows])
fig,ax=plt.subplots(1,3,figsize=(14,4),constrained_layout=True)
for key,label in [('gram_error','K'),('weight_error','X'),('delivered_functional_error','delivered functional')]:
    ax[0].loglog(h,[r[key] for r in rows],'o-',label=label)
ax[0].legend(fontsize=8);ax[0].set_title('Approach to the continuum calculation')
for k,(color,label) in enumerate(zip(COLORS,F.LABELS)):
    ax[1].semilogx(h,[r['estimate'][k] for r in rows],'o-',color=color,label=label)
    ax[1].axhline(reference['estimate'][k],color=color,ls='--')
    ax[1].axhline(ex.truth[k],color=color,ls=':',alpha=.6)
ax[1].legend(fontsize=8);ax[1].set_title('Dashed continuum estimate; dotted target truth')
ax[2].loglog(h,[r['projected_response_error'] for r in rows],'o-')
ax[2].set_title('Projected normalization error\n(actual response is one throughout)')
for a in ax:a.invert_xaxis();a.set_xlabel('trial-centre spacing h (finer to the right)')
plt.show()


## Physical questions and full continuum response maps

The top row shows the physical measures: an exact ray integral, a uniform disk
average, and the signed weighted rays selected by SOLA. A ray has zero area but
nonzero measure mass. The disk has density $1/(\pi R^2)$ inside its boundary.
The delivered measure has mass $\sum_i x_iL_i=1$ and retains its ray support.

For the bottom row, place the **same known unit-peak smooth test anomaly** at each
centre $z$ and ask what these physical measurements return:
$$G_iq_z,\qquad T_aq_z,\qquad (XG)_a q_z.$$
The test shape and its horizontal profile are shown explicitly. Its chosen
length is 0.1 throughout; it is neither a physical ray width nor information
about the unknown model. It has decaying tails, rather than compact support.
The heatmaps are **responses to test models**, not area-density representations
of the ray measures. Ray responses have units of model times length; target and
delivered averages have model units, with a shared colour scale for that pair.

For this H2 space, the test model is the unit-peak reproducing kernel
$$q_z(y)=\frac{K_H(y,z)}{K_H(z,z)},\qquad
(G_iq_z,T_aq_z,(X_hG)_a q_z)
=\frac{(g_i(z),t_a(z),(G^*x_{h,a})(z))}{K_H(z,z)}.$$
The smooth maps therefore have a direct physical reading without changing the
observations: they predict measurements of this specified test model.

The evolution below is the **full continuum response** $X_hGq_z$. The following
projected-field section instead shows $P_hG^*x_h$, whose rescaled values are
$X_hG(P_hq_z)$. Those agree on trial models but generally differ on arbitrary
continuum test models. Keeping both views exposes the trial approximation.

In [ ]:
# physical-probe: galerkin
import probe_views as V
probes=V.response_grid(ex,OUT)
V.plot_probe(ex,probes);plt.show()
V.plot_physical_and_responses(ex,probes,weights[-1][0],
    f"H2 Galerkin ({rows[-1]['n']} per axis): actual measures and responses");plt.show()
indices=[i for i,r in enumerate(rows) if r['n'] in {3,9,17,rows[-1]['n']}]
V.plot_response_evolution(ex,probes,[weights[i][0] for i in indices],
    [f"{rows[i]['n']} x {rows[i]['n']} trial centres" for i in indices],
    'Full continuum responses X_h G q_z (trial projection shown separately below)',
    reference=Xref[0]);plt.show()
print('Actual responses use G q_z, not G P_h q_z.')


## Watch the finite resolving representer converge

For the first disk, plot the actual **finite-space** resolving representer
$$r_h=P_hG^*x_h=\sum_j c_{h,j}\phi_j,
\qquad c_h=M_h^{-1}E_h^Tx_h.$$
This includes both the changing SOLA weights and the trial-space projection.
Plotting only $G^*x_h$ would omit the latter.

All panels use the same evaluation grid and colour scale, with the requested
disk outlined. The last panels show the native limit $G^*x$ and the target Riesz
representer $t_H$. These are **H2 Riesz functions, not area-density kernels**:
their interpretation is through the H2 inner product. Their displayed area
integrals need not be one. The physical response $x_h^TG1$ remains one, while
$\langle r_h,1\rangle_H=x_h^TGP_h1$ is the projected response reported above.

The extra assembly below is an untimed plotting diagnostic. Small blocks bound
temporary kernel arrays; these computations do not change the benchmark.

In [ ]:
plot_n=60
z=(np.arange(plot_n)+.5)/plot_n
xx,yy=np.meshgrid(z,z)
plot_points=np.column_stack([xx.ravel(),yy.ravel()])
selected=[(r['n'],W[0]) for r,W in zip(rows,weights)
          if r['n'] in {3,9,17,rows[-1]['n']}]
pictures={};field_errors={}
block=64
native_norm2=float(Xref[0]@ref['K']@Xref[0])
for n,x_h in selected:
    z_h=np.linspace(0,1,n);xh,yh=np.meshgrid(z_h,z_h)
    trial_points=np.column_stack([xh.ravel(),yh.ravel()])
    Phi_h=H.riesz(trial_points)
    m=n*n
    M_plot=np.empty((m,m),order='F');E_plot=np.empty((len(ex.d),m))
    with S.quadrature_nodes(24):
        for j in range(0,m,block):
            stop=min(j+block,m)
            # Fresh families keep the atom sets small; slicing a Family may
            # retain the full atom set and merely select coefficient columns.
            trial_block=H.riesz(trial_points[j:stop])
            M_plot[:,j:stop]=H.inner(Phi_h,trial_block)
            E_plot[:,j:stop]=H.inner(F_ray,trial_block)
    rhs=E_plot.T@x_h
    # Overwrite this diagnostic mass matrix to avoid a second full matrix copy.
    factor_plot=la.cho_factor(M_plot,lower=True,overwrite_a=True,check_finite=False)
    c_h=la.cho_solve(factor_plot,rhs,check_finite=False)
    error2=float(c_h@rhs-2*c_h@(E_plot.T@Xref[0])+native_norm2)
    field_errors[n]=float(np.sqrt(max(error2,0)/native_norm2))
    values=np.empty(len(plot_points))
    for j in range(0,len(values),block):
        stop=min(j+block,len(values))
        points=H.riesz(plot_points[j:stop])
        values[j:stop]=np.asarray(H.inner(points,Phi_h))@c_h
    pictures[n]=values.reshape(plot_n,plot_n)
    del M_plot,E_plot,factor_plot,Phi_h
    print(f'{n} x {n}: relative H2 representer error = {field_errors[n]:.3e}')

native_values=np.empty(len(plot_points));target_values=np.empty(len(plot_points))
with S.quadrature_nodes(24):
    for j in range(0,len(plot_points),block):
        stop=min(j+block,len(plot_points));points=H.riesz(plot_points[j:stop])
        native_values[j:stop]=np.asarray(H.inner(points,F_ray))@Xref[0]
        target_values[j:stop]=np.asarray(H.inner(points,F_target[0])).reshape(-1)
native_values=native_values.reshape(plot_n,plot_n)
target_values=target_values.reshape(plot_n,plot_n)
panels=[(f'{n} x {n} trial centres\nrelative H2 error {field_errors[n]:.2e}',v)
        for n,v in pictures.items()]
panels += [('Native continuum resolving representer',native_values),
           ('Uniform-disk target: H2 representer',target_values)]
lim=max(float(np.max(np.abs(v))) for _,v in panels)
fig,axes=plt.subplots(1,len(panels),figsize=(3.2*len(panels),3.7),constrained_layout=True)
for ax,(title,values) in zip(np.atleast_1d(axes),panels):
    im=ax.imshow(values,origin='lower',extent=[0,1,0,1],cmap='RdBu_r',vmin=-lim,vmax=lim)
    ax.add_patch(plt.Circle(ex.centres[0],ex.radius,fill=False,color='k'))
    ax.set_title(title,fontsize=9);ax.set_xticks([]);ax.set_yticks([]);ax.grid(False)
fig.colorbar(im,ax=np.atleast_1d(axes),shrink=.7,label='H2 Riesz function; common colour scale')
plt.show()
F.save(OUT/'galerkin_fields.json',dict(levels=list(pictures),relative_H2_errors=field_errors,
       target_index=0,plot_grid=plot_n,interpretation='P_h G* x_h, paired in H2; not area density',
       plot_host=socket.gethostname(),threads=threadpool_info()))
np.savez_compressed(OUT/'galerkin_fields.npz',levels=list(pictures),fields=np.array(list(pictures.values())),
                    native=native_values,target=target_values,points=plot_points)
assert field_errors[selected[-1][0]]<field_errors[selected[0][0]]
assert all(np.all(np.isfinite(v)) for _,v in panels)


## What this establishes

Discretizing first works when it is a consistent approximation of the declared
continuum formulation. There is no need to avoid finite matrices. The choice of
H2 and its metric fixes the SOLA comparison geometry; it does not assert an upper
bound on the unknown model norm. The limiting question remains $XG$, not exactly T.

This particular finite realization uses **dense kernel-translate trial matrices**.
Sparse finite elements or other conforming bases may have different costs. The
next notebook computes the same continuum objective directly in data space.
The fixed 24-node/32-chord integration error remains separate from trial error.